In [ ]:
#Part 4 — LLM API Integration

In [ ]:
import getpass
import os

api_input = getpass.getpass("Enter your OpenRouter API Key securely: ")
os.environ["LLM_API_KEY"] = api_input

print("LLM_API_KEY has been successfully set in the session memory!")

In [ ]:
import os
import re
import json
import joblib
import requests
import jsonschema
from jsonschema import validate

print("All required standard libraries and jsonschema modules imported successfully.")

In [ ]:
import os

# Fetching the API token dynamically from the environment configuration
API_KEY = os.environ.get("LLM_API_KEY")
API_URL = "https://openrouter.ai/api/v1/chat/completions"
MODEL_NAME = "openrouter/auto"  # Optimized to use OpenRouter's auto-router to avoid 401 errors

# Defining strict target JSON schema with 5 required scalar fields as requested
EXPLANATION_SCHEMA = {
    "type": "object",
    "properties": {
        "prediction_label": {"type": "string"},
        "confidence_level": {"type": "string"},
        "top_reason": {"type": "string"},
        "second_reason": {"type": "string"},
        "next_step": {"type": "string"}
    },
    "required": ["prediction_label", "confidence_level", "top_reason", "second_reason", "next_step"]
}

# Fallback dictionary to keep the pipeline alive if validation or JSON decoding fails
FALLBACK_DICT = {
    "prediction_label": "null",
    "confidence_level": "null",
    "top_reason": "null",
    "second_reason": "null",
    "next_step": "null"
}

print("API configurations updated with openrouter/auto and schema defined.")

In [ ]:
#Safety Guardrails (PII Check Function)
def has_pii(text):
    """
    Scans the given user prompt utilizing regular expressions to actively detect
    personally identifiable information such as email strings or 10-digit phone layouts.
    """
    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'
    return bool(re.search(email_pattern, text) or re.search(phone_pattern, text))

print("PII Guardrail engine successfully declared and ready for validation.")

In [ ]:
#Reusable LLM Connector Function
def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):
    """
    Establishes a secure connection to the LLM API using OpenRouter automated routing.
    Fetches the API token dynamically from environment variables to avoid hardcoding.
    """
    # Dynamically extract active session tokens from process environment
    current_key = os.environ.get("LLM_API_KEY")
    if not current_key:
        print("Backend Error: LLM_API_KEY environment variable is missing.")
        return None
        
    # Standard mandatory structural headers required by OpenRouter
    headers = {
        "Authorization": f"Bearer {current_key}",
        "Content-Type": "application/json"
    }
    
    # Pack parameters inside payload optimizing automated model distribution
    payload = {
        "model": "openrouter/auto",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }
    
    try:
        # Fire robust POST request out to endpoint
        response = requests.post("https://openrouter.ai/api/v1/chat/completions", headers=headers, json=payload, timeout=15)
        
        # Verify response state before returning textual content choice
        if response.status_code == 200:
            return response.json()['choices'][0]['message']['content']
        else:
            print(f"API Connection Rejected! Returned HTTP Status Code: {response.status_code}")
            return None
    except Exception as e:
        print(f"Network Request Failure: Unexpected execution anomaly caught: {e}")
        return None

print(" Engine initialized successfully: call_llm wrapper established.")

In [ ]:
#Core Functionality Verification
#  verification task 1: Basic Connection Test
print("--- TASK 1: Verifying Basic Connection Test ---")
test_res = call_llm("You are a helpful assistant.", "Reply with only the word: hello")
print(f"Response from LLM: {test_res}\n")

# verification task 2: PII Guardrail Test Cases 
print("--- TASK 2: Testing PII Guardrail Verification ---")
pii_input = "Please review data for customer fake_email@gmail.com or ring 9876543210."
clean_input = "Passenger vectors: Pclass 3, Male, Age 22, Fare 7.25"

for idx, current_test in enumerate([pii_input, clean_input], 1):
    print(f"Evaluating Scenario {idx}: '{current_test}'")
    if has_pii(current_test):
        print("Outcome -> Input blocked: PII detected.\n")
    else:
        print("Outcome -> Guardrail passed. Ready to forward to LLM call.\n")

In [ ]:
# verification task 1: Basic Connection Test ---
print("--- TASK 1: Verifying Basic Connection Test ---")
test_res = call_llm("You are a helpful assistant.", "Reply with only the word: hello")
print(f"Response from LLM: {test_res}\n")

# verification task 2: PII Guardrail Test Cases ---
print("--- TASK 2: Testing PII Guardrail Verification ---")
pii_input = "Please review data for customer fake_email@gmail.com or ring 9876543210."
clean_input = "Passenger vectors: Pclass 3, Male, Age 22, Fare 7.25"

# Process both structural variants through regex matching framework
for idx, current_test in enumerate([pii_input, clean_input], 1):
    print(f"Evaluating Scenario {idx}: '{current_test}'")
    if has_pii(current_test):
        print("Outcome -> Input blocked: PII detected. Pipeline halted for security constraints.\n")
    else:
        print("Outcome -> Guardrail passed. Input clean and ready to forward to LLM call.\n")

In [ ]:
def process_and_validate_response(raw_response):
    """
    Isolates embedded JSON sub-blocks, formats strings into clean dictionary structures,
    and runs a rigid validation schema check catching architectural mismatches.
    """
    if not raw_response:
        return FALLBACK_DICT
        
    try:
        clean_response = raw_response.strip()
        
        # Handle standard backtick formatting wrapper variables dynamically
        if clean_response.startswith("```json"):
            clean_response = clean_response.split("```json")[1].split("```")[0].strip()
        elif clean_response.startswith("```"):
            clean_response = clean_response.split("```")[1].split("```")[0].strip()
            
        # Parse clean alphanumeric string into actual JSON dictionary mapping object
        parsed_json = json.loads(clean_response)
        
        # Explicit validation check executing against pre-defined 5-scalar scalar keys
        validate(instance=parsed_json, schema=EXPLANATION_SCHEMA)
        return parsed_json
        
    except json.JSONDecodeError as je:
        print(f"JSON Decode Error triggered during raw processing: {je}")
        return FALLBACK_DICT
    except jsonschema.ValidationError as ve:
        print(f"Schema Validation Failure occurred: {ve.message}")
        return FALLBACK_DICT

print(" Validation mapping functions defined.")

In [ ]:
print("--- TASK 3: Executing End-to-End Prediction Explanation Pipeline ---\n")

# Array storing 3 specific handcrafted feature-vectors mapping distinct passenger archetypes
hand_crafted_passengers = [
    {"Pclass": 1, "Sex": "female", "Age": 29, "SibSp": 0, "Parch": 0, "Fare": 211.33, "Embarked": "S"},
    {"Pclass": 3, "Sex": "male", "Age": 22, "SibSp": 1, "Parch": 0, "Fare": 7.25, "Embarked": "S"},
    {"Pclass": 2, "Sex": "male", "Age": 2, "SibSp": 1, "Parch": 1, "Fare": 26.00, "Embarked": "S"}
]

# Strict system context driving rigid layout format instructions
zero_shot_system_prompt = (
    "You are a model prediction explainer. Given feature values, a predicted class, and a predicted probability "
    "for a Titanic passenger survival model, output ONLY a valid JSON object matching the requested schema. "
    "Do not include any introductory or concluding markdown conversational text. Output fields must be exactly: "
    "prediction_label, confidence_level, top_reason, second_reason, next_step."
)

# Simulated model classifier reproducing structured prediction inferences
def execute_model_inference(passenger):
    if passenger["Pclass"] == 1 and passenger["Sex"] == "female":
        return "Survived", 0.95
    elif passenger["Pclass"] == 3 and passenger["Sex"] == "male":
        return "Not Survived", 0.88
    else:
        return "Survived", 0.65

# Main orchestration processing records through structural gates, predictions, and temperature variations
for index, passenger in enumerate(hand_crafted_passengers, 1):
    # Step A: Capture specific classification vectors from the inference engine
    pred_label, pred_probability = execute_model_inference(passenger)
    
    # Step B: Construct payload strings targeting model explanation contexts
    user_prompt_payload = (
        f"Passenger Features: {json.dumps(passenger)}\n"
        f"Predicted Class: {pred_label}\n"
        f"Probability: {pred_probability}"
    )
    
    print(f"========================================= PASSENGER TARGET RECORD {index} =========================================")
    print(f"Input Vector Features: {passenger}")
    print(f"Model Inferences: Classified as '{pred_label}' | Certainty Score: {pred_probability}\n")
    
    # Step C: Proactive PII scan execution before initiating outside communication
    if has_pii(user_prompt_payload):
        print("Input blocked: PII detected. Pipeline halted for safety constraints.\n")
        continue
    else:
        print("Guardrail passed: Input contains no sensitive PII details. Safe to proceed.")
        
    # Step D: Execute Temperature comparison evaluation loop (A/B testing)
    for current_temp in [0.0, 0.7]:
        print(f"\n--- Triggering Inference Sequence at Temperature setting: {current_temp} ---")
        raw_llm_output = call_llm(zero_shot_system_prompt, user_prompt_payload, temperature=current_temp)
        print(f"Raw Output Received:\n{raw_llm_output}")
        
        # Run deep data schema verification layout only for temperature 0.0 production runs
        if current_temp == 0.0:
            validated_json_dict = process_and_validate_response(raw_llm_output)
            print(f"\nValidated Scheme-Compliant Dictionary Object:")
            print(json.dumps(validated_json_dict, indent=2))
    print("\n")

In [ ]:
# ==========================================
# INTERACTIVE USER INPUT PREDICTION SYSTEM
# ==========================================

print("=========================================")
print("     TITANIC SURVIVAL PREDICTION SYSTEM  ")
print("=========================================\n")

# 1. Collecting live data from the user
name = input("Enter Passenger Name: ")
pclass = int(input("Enter Ticket Class (1, 2, or 3): "))
sex = input("Enter Gender (male or female): ").strip().lower()
age = float(input("Enter Age: "))
sibsp = int(input("Enter Siblings / Spouses count: "))
parch = int(input("Enter Parents / Children count: "))
fare = float(input("Enter Ticket Fare (e.g., 50.0): "))
embarked = input("Enter Embarked Port (S, C, or Q): ").strip().upper()

# 2. Transforming input data into a DataFrame format
user_passenger_data = {
    'Pclass': [pclass],
    'Age': [age],
    'SibSp': [sibsp],
    'Parch': [parch],
    'Fare': [fare],
    'Sex_male': [1 if sex == 'male' else 0],
    'Embarked_Q': [1 if embarked == 'Q' else 0],
    'Embarked_S': [1 if embarked == 'S' else 0]
}

df_user = pd.DataFrame(user_passenger_data)

# Align columns with the training dataset features (X)
df_user = df_user.reindex(columns=X.columns, fill_value=0)

# 3. Executing Model Prediction and Probabilities
prediction = model.predict(df_user)[0]
probability = model.predict_proba(df_user)[0]

# 4. Final Result Output Display
print("\n=========================================")
print(f"      PREDICTION FOR: {name.upper()}       ")
print("=========================================")

if prediction == 1:
    confidence = probability[1] * 100
    print(f"Status    : SURVIVED 🎉")
    print(f"Certainty : {confidence:.2f}%")
else:
    confidence = probability[0] * 100
    print(f"Status    : NOT SURVIVED ❌")
    print(f"Certainty : {confidence:.2f}%")

print("=========================================")